<sub>**The Annotated ML Handbook** · Part V: scikit-learn · by Mahir Absar Khan</sub>

# Chapter 20: Preprocessing, feature engineering and pipelines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-5-scikit-learn/20-sklearn-preprocessing.ipynb) &nbsp; [📚 Contents](../../book/README.md) &nbsp; [🌐 Web edition](https://mahir-606.github.io/annotated-ml-handbook/)

*Run the setup cell first, then read and run the cells from top to bottom. Every code cell is followed by a table explaining what each line does.*

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

Models need numbers on sensible scales, with no missing values. Real tables have text categories, gaps and wildly different units. scikit-learn's **transformers** fix each of these problems, and a **Pipeline** chains them with a model so the whole thing is fitted correctly, without leakage, in one call.

## 20.1 Scaling numeric features · *Beginner*

> **📖 Definitions**
>
> **Standardisation (z-scoring)**: rescaling a feature to mean 0 and standard deviation 1: (x − mean) / SD.
>
> **Normalisation (min-max scaling)**: rescaling a feature to a fixed range, usually 0–1: (x − min) / (max − min).
>
> **Transformer (scikit-learn)**: an estimator with `fit` and `transform` that changes data, such as a scaler, encoder or imputer.


Distance-based and gradient-based models (kNN, SVM, logistic/linear regression with regularisation, neural networks, PCA) are sensitive to feature scale. Tree-based models (decision trees, random forests, gradient boosting) are not.

**StandardScaler, MinMaxScaler, RobustScaler**

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

X = np.array([[1.0, 200.0], [2.0, 300.0], [3.0, 250.0], [4.0, 5000.0]])
for scaler in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    Xs = scaler.fit_transform(X)
    print(f"{type(scaler).__name__:<15}", Xs[:, 1].round(2))
sc = StandardScaler().fit(X)
print("learned mean_:", sc.mean_, " scale_:", sc.scale_.round(1))
print(sc.inverse_transform(sc.transform(X))[0])

StandardScaler  [-0.6  -0.55 -0.58  1.73]
MinMaxScaler    [0.   0.02 0.01 1.  ]
RobustScaler    [-0.06  0.02 -0.02  3.82]
learned mean_: [   2.5 1437.5]  scale_: [1.1000e+00 2.0571e+03]
[  1. 200.]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import numpy as np</code></td><td>Import NumPy (the array library) under the short name <code>np</code>.</td></tr>
<tr><td><code>import pandas as pd</code></td><td>Import pandas (the table library) under the short name <code>pd</code>.</td></tr>
<tr><td><code>from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler</code></td><td>Import <code>StandardScaler</code>, <code>MinMaxScaler</code> and <code>RobustScaler</code> from scikit-learn&#x27;s <code>preprocessing</code> module.</td></tr>
<tr><td><code>X = np.array([[1.0, 200.0], [2.0, 300.0], [3.0, 250.0], [4.0, 5000.0]])</code></td><td>Column 2 has an outlier (5000).</td></tr>
<tr><td><code>for scaler in [StandardScaler(), MinMaxScaler(), RobustScaler()]:</code></td><td>Loop over <code>[StandardScaler(), MinMaxScaler(), RobustScaler()]</code>, naming each item <code>scaler</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;Xs = scaler.fit_transform(X)</code></td><td>Learn the statistics from X, then transform X.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;print(f&quot;{type(scaler).__name__:&lt;15}&quot;, Xs[:, 1].round(2))</code></td><td>The scaler&#x27;s class name and the scaled second column.</td></tr>
<tr><td><code>sc = StandardScaler().fit(X)</code></td><td>Fit a scaler to inspect what it learned.</td></tr>
<tr><td><code>print(&quot;learned mean_:&quot;, sc.mean_, &quot; scale_:&quot;, sc.scale_.round(1))</code></td><td>Learned statistics end with &quot;_&quot;.</td></tr>
<tr><td><code>print(sc.inverse_transform(sc.transform(X))[0])</code></td><td>inverse_transform undoes the scaling.</td></tr>
</table>
</details>

| Scaler | Formula | Use when |
|---|---|---|
| `StandardScaler` | (x − mean) / sd | default choice; roughly symmetric features |
| `MinMaxScaler` | (x − min) / (max − min) | you need a bounded range [0, 1], e.g. pixel intensities |
| `RobustScaler` | (x − median) / IQR | features with outliers |
| `PowerTransformer` / `np.log1p` | makes skewed data more normal | strongly right-skewed features (incomes, counts) |

> **⚠️ Pitfall** · **Fit on training data, transform both**
>
> `scaler.fit_transform(X_train)` then `scaler.transform(X_test)`. Never `fit` on the test set, and never fit on all data before splitting. A Pipeline enforces this for you.

## 20.2 Encoding categorical features · *Beginner*

**One-hot and ordinal encoding**

In [3]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

df = pd.DataFrame({"region": ["north", "south", "east", "south"],
                   "size": ["S", "L", "M", "M"]})
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
print(ohe.fit_transform(df[["region"]]))
print(ohe.get_feature_names_out())
print(ohe.transform(pd.DataFrame({"region": ["west"]})))
ord_enc = OrdinalEncoder(categories=[["S", "M", "L"]])
print(ord_enc.fit_transform(df[["size"]]).ravel())

[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 0. 1.]]
['region_east' 'region_north' 'region_south']
[[0. 0. 0.]]
[0. 2. 1. 1.]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder</code></td><td>Import <code>OneHotEncoder</code> and <code>OrdinalEncoder</code> from scikit-learn&#x27;s <code>preprocessing</code> module.</td></tr>
<tr><td><code>df = pd.DataFrame({&quot;region&quot;: [&quot;north&quot;, &quot;south&quot;, &quot;east&quot;, &quot;south&quot;],</code></td><td>A small table with an unordered category...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&quot;size&quot;: [&quot;S&quot;, &quot;L&quot;, &quot;M&quot;, &quot;M&quot;]})</code></td><td>...and an ordered one.</td></tr>
<tr><td><code>ohe = OneHotEncoder(sparse_output=False, handle_unknown=&quot;ignore&quot;)</code></td><td>handle_unknown=&quot;ignore&quot;: an unseen category becomes all zeros instead of an error.</td></tr>
<tr><td><code>print(ohe.fit_transform(df[[&quot;region&quot;]]))</code></td><td>One 0/1 column per category (alphabetical order).</td></tr>
<tr><td><code>print(ohe.get_feature_names_out())</code></td><td>The names of the new columns, e.g. region_east.</td></tr>
<tr><td><code>print(ohe.transform(pd.DataFrame({&quot;region&quot;: [&quot;west&quot;]})))</code></td><td>&quot;west&quot; was never seen during fit: all zeros.</td></tr>
<tr><td><code>ord_enc = OrdinalEncoder(categories=[[&quot;S&quot;, &quot;M&quot;, &quot;L&quot;]])</code></td><td>State the ORDER explicitly for ordered categories.</td></tr>
<tr><td><code>print(ord_enc.fit_transform(df[[&quot;size&quot;]]).ravel())</code></td><td>S=0, M=1, L=2.</td></tr>
</table>
</details>

> **💡 Tip** · **Which encoder?**
>
> Use **one-hot** for unordered categories (region, colour), **ordinal** for genuinely ordered ones (S < M < L, education level). Tree models also cope well with ordinal codes for unordered categories. For very high-cardinality columns (thousands of postcodes), consider `TargetEncoder` (it is cross-fitted internally to limit leakage) or grouping rare levels with `OneHotEncoder(min_frequency=...)`.

## 20.3 Imputing missing values · *Beginner*

**SimpleImputer and KNNImputer**

In [4]:
from sklearn.impute import SimpleImputer, KNNImputer

Xm = np.array([[1.0, 10.0], [np.nan, 12.0], [3.0, np.nan], [4.0, 16.0]])
print(SimpleImputer(strategy="median").fit_transform(Xm))
print(SimpleImputer(strategy="mean", add_indicator=True).fit_transform(Xm))
print(KNNImputer(n_neighbors=2).fit_transform(Xm).round(2))
print(SimpleImputer(strategy="most_frequent").fit_transform(np.array([["a"], ["b"], ["a"], [np.nan]], dtype=object)).ravel())

[[ 1. 10.]
 [ 3. 12.]
 [ 3. 12.]
 [ 4. 16.]]
[[ 1.         10.          0.          0.        ]
 [ 2.66666667 12.          1.          0.        ]
 [ 3.         12.66666667  0.          1.        ]
 [ 4.         16.          0.          0.        ]]
[[ 1.  10. ]
 [ 2.5 12. ]
 [ 3.  13. ]
 [ 4.  16. ]]
['a' 'b' 'a' 'a']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from sklearn.impute import SimpleImputer, KNNImputer</code></td><td>Import <code>SimpleImputer</code> and <code>KNNImputer</code> from scikit-learn&#x27;s <code>impute</code> module.</td></tr>
<tr><td><code>Xm = np.array([[1.0, 10.0], [np.nan, 12.0], [3.0, np.nan], [4.0, 16.0]])</code></td><td>Two missing values.</td></tr>
<tr><td><code>print(SimpleImputer(strategy=&quot;median&quot;).fit_transform(Xm))</code></td><td>Replace NaN with the column median (learned on fit).</td></tr>
<tr><td><code>print(SimpleImputer(strategy=&quot;mean&quot;, add_indicator=True).fit_transform(Xm))</code></td><td>add_indicator appends 0/1 &quot;was missing&quot; columns.</td></tr>
<tr><td><code>print(KNNImputer(n_neighbors=2).fit_transform(Xm).round(2))</code></td><td>Fill from the 2 most similar rows.</td></tr>
<tr><td><code>print(SimpleImputer(strategy=&quot;most_frequent&quot;).fit_transform(np.array([[&quot;a&quot;], [&quot;b&quot;], [&quot;a&quot;], [np.nan]], dtype=object)).ravel())</code></td><td>For categories.</td></tr>
</table>
</details>

> **📝 Note** · **Models that accept NaN directly**
>
> `HistGradientBoostingClassifier/Regressor`, `DecisionTree*`, and `RandomForest*` in recent scikit-learn versions handle missing values natively. Most other estimators raise `ValueError: Input X contains NaN`.

## 20.4 Pipelines: preprocessing + model as one object · *Beginner*

> **📖 Definitions**
>
> **Pipeline**: a chain of preprocessing steps and a final model that is fitted and applied as one estimator, guaranteeing every step learns only from training data.


A `Pipeline` is a list of steps. Calling `fit` runs `fit_transform` on each transformer, in order, then `fit` on the final model. Calling `predict` runs `transform` through each step, then `predict`. The fitted statistics always come from the data passed to `fit`.

**A simple pipeline**

In [5]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)

pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_tr, y_tr)
print("test accuracy:", round(pipe.score(X_te, y_te), 4))
print(pipe.named_steps["scale"].mean_[:3].round(2))
quick = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
print([name for name, _ in quick.steps])

test accuracy: 0.958
[14.12 19.27 91.91]
['standardscaler', 'logisticregression']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from sklearn.pipeline import Pipeline, make_pipeline</code></td><td>Import <code>Pipeline</code> and <code>make_pipeline</code> from scikit-learn&#x27;s <code>pipeline</code> module.</td></tr>
<tr><td><code>from sklearn.linear_model import LogisticRegression</code></td><td>Import <code>LogisticRegression</code> from scikit-learn&#x27;s <code>linear_model</code> module.</td></tr>
<tr><td><code>from sklearn.datasets import load_breast_cancer</code></td><td>Import <code>load_breast_cancer</code> from scikit-learn&#x27;s <code>datasets</code> module.</td></tr>
<tr><td><code>from sklearn.model_selection import train_test_split</code></td><td>Import <code>train_test_split</code> from scikit-learn&#x27;s <code>model_selection</code> module.</td></tr>
<tr><td><code>X, y = load_breast_cancer(return_X_y=True, as_frame=True)</code></td><td>569 tumours, 30 numeric features, target 0 = malignant, 1 = benign.</td></tr>
<tr><td><code>X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)</code></td><td>Stratified 75/25 split.</td></tr>
<tr><td><code>pipe = Pipeline([</code></td><td>A Pipeline is built from a list of (name, step) pairs.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;(&quot;impute&quot;, SimpleImputer(strategy=&quot;median&quot;)),</code></td><td>Step name + transformer object.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;(&quot;scale&quot;, StandardScaler()),</code></td><td>Then standardise.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;(&quot;clf&quot;, LogisticRegression(max_iter=1000)),</code></td><td>The last step is the model.</td></tr>
<tr><td><code>])</code></td><td>Close the list and the call or tuple opened above.</td></tr>
<tr><td><code>pipe.fit(X_tr, y_tr)</code></td><td>Each step is fitted on training data only.</td></tr>
<tr><td><code>print(&quot;test accuracy:&quot;, round(pipe.score(X_te, y_te), 4))</code></td><td>The pipeline scores like a model.</td></tr>
<tr><td><code>print(pipe.named_steps[&quot;scale&quot;].mean_[:3].round(2))</code></td><td>Reach inside to inspect a fitted step.</td></tr>
<tr><td><code>quick = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))</code></td><td>Same idea; step names are generated automatically.</td></tr>
<tr><td><code>print([name for name, _ in quick.steps])</code></td><td>Automatic names: lowercase class names.</td></tr>
</table>
</details>

## 20.5 ColumnTransformer: different processing per column · *Intermediate*

> **📖 Definitions**
>
> **ColumnTransformer**: a scikit-learn tool that applies different preprocessing to different groups of columns and joins the results.


Real tables mix numeric and categorical columns. `ColumnTransformer` routes each group of columns through its own mini-pipeline and concatenates the results.

**The standard tabular preprocessing recipe**

In [6]:
from sklearn.compose import ColumnTransformer, make_column_selector

ins = pd.read_csv("insurance.csv")
X = ins.drop(columns="charges")
y = ins["charges"]
num_cols = ["age", "bmi", "children"]
cat_cols = ["sex", "smoker", "region"]

numeric = Pipeline([("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler())])
categorical = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))])
pre = ColumnTransformer([
    ("num", numeric, num_cols),
    ("cat", categorical, cat_cols),
], remainder="drop")

Xt = pre.fit_transform(X)
print(Xt.shape)
print(pre.get_feature_names_out())

(1000, 9)
['num__age' 'num__bmi' 'num__children' 'cat__sex_male' 'cat__smoker_yes'
 'cat__region_northeast' 'cat__region_northwest' 'cat__region_southeast'
 'cat__region_southwest']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from sklearn.compose import ColumnTransformer, make_column_selector</code></td><td>Import <code>ColumnTransformer</code> and <code>make_column_selector</code> from scikit-learn&#x27;s <code>compose</code> module.</td></tr>
<tr><td><code>ins = pd.read_csv(&quot;insurance.csv&quot;)</code></td><td>The Part III dataset (contains missing BMI).</td></tr>
<tr><td><code>X = ins.drop(columns=&quot;charges&quot;)</code></td><td>Features: every column except the target.</td></tr>
<tr><td><code>y = ins[&quot;charges&quot;]</code></td><td>Target.</td></tr>
<tr><td><code>num_cols = [&quot;age&quot;, &quot;bmi&quot;, &quot;children&quot;]</code></td><td>Numeric columns.</td></tr>
<tr><td><code>cat_cols = [&quot;sex&quot;, &quot;smoker&quot;, &quot;region&quot;]</code></td><td>Categorical columns.</td></tr>
<tr><td><code>numeric = Pipeline([(&quot;impute&quot;, SimpleImputer(strategy=&quot;median&quot;)),</code></td><td>Numeric branch: fill gaps with the median...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(&quot;scale&quot;, StandardScaler())])</code></td><td>...then standardise.</td></tr>
<tr><td><code>categorical = Pipeline([(&quot;impute&quot;, SimpleImputer(strategy=&quot;most_frequent&quot;)),</code></td><td>Categorical branch: fill gaps with the most frequent value...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;(&quot;onehot&quot;, OneHotEncoder(handle_unknown=&quot;ignore&quot;, drop=&quot;if_binary&quot;))])</code></td><td>drop=&quot;if_binary&quot;: one column for yes/no features.</td></tr>
<tr><td><code>pre = ColumnTransformer([</code></td><td>Route each column group to its branch:</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;(&quot;num&quot;, numeric, num_cols),</code></td><td>(name, transformer, columns)</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;(&quot;cat&quot;, categorical, cat_cols),</code></td><td>Categorical columns through the categorical branch.</td></tr>
<tr><td><code>], remainder=&quot;drop&quot;)</code></td><td>Columns not listed are dropped (&quot;passthrough&quot; keeps them).</td></tr>
<tr><td><code>Xt = pre.fit_transform(X)</code></td><td>Fit both branches and transform: a numeric matrix comes out.</td></tr>
<tr><td><code>print(Xt.shape)</code></td><td>Display the value of <code>Xt.shape</code>.</td></tr>
<tr><td><code>print(pre.get_feature_names_out())</code></td><td>Track exactly which column is which after transformation.</td></tr>
</table>
</details>

**Plugging it into a model**

In [7]:
from sklearn.linear_model import Ridge
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
model = Pipeline([("pre", pre), ("reg", Ridge(alpha=1.0))])
model.fit(X_tr, y_tr)
print("test R2:", round(model.score(X_te, y_te), 4))
new_customer = pd.DataFrame([{"age": 45, "sex": "female", "bmi": np.nan, "children": 2,
                              "smoker": "no", "region": "southwest"}])
print("predicted charges:", model.predict(new_customer).round(0))

test R2: 0.9311
predicted charges: [19086.]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from sklearn.linear_model import Ridge</code></td><td>Import <code>Ridge</code> from scikit-learn&#x27;s <code>linear_model</code> module.</td></tr>
<tr><td><code>X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)</code></td><td>80/20 split of the raw table.</td></tr>
<tr><td><code>model = Pipeline([(&quot;pre&quot;, pre), (&quot;reg&quot;, Ridge(alpha=1.0))])</code></td><td>A pipeline inside a pipeline.</td></tr>
<tr><td><code>model.fit(X_tr, y_tr)</code></td><td>Raw DataFrame in: imputing, scaling and encoding happen inside.</td></tr>
<tr><td><code>print(&quot;test R2:&quot;, round(model.score(X_te, y_te), 4))</code></td><td>R² on the held-out test set.</td></tr>
<tr><td><code>new_customer = pd.DataFrame([{&quot;age&quot;: 45, &quot;sex&quot;: &quot;female&quot;, &quot;bmi&quot;: np.nan, &quot;children&quot;: 2,</code></td><td>A new raw record, including a missing BMI...</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&quot;smoker&quot;: &quot;no&quot;, &quot;region&quot;: &quot;southwest&quot;}])</code></td><td>...continued.</td></tr>
<tr><td><code>print(&quot;predicted charges:&quot;, model.predict(new_customer).round(0))</code></td><td>Works on raw new data, missing BMI included.</td></tr>
</table>
</details>

> **💡 Tip** · **Select columns by type automatically**
>
> `make_column_selector(dtype_include="number")` and `make_column_selector(dtype_include=object)` pick columns by dtype, so the preprocessor adapts when columns are added. In pandas 3, string columns match `dtype_include=["object", "string"]`.

## 20.6 Seeing leakage with your own eyes · *Intermediate*

> **📖 Definitions**
>
> **Data leakage**: information from outside the training data (often from the test set or the target itself) influencing model training, which produces overly optimistic results.


Here is why pipelines matter. We select the "best" features using *all* the data, then cross-validate on pure noise.

**Feature selection outside vs inside the pipeline**

In [8]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(100, 5000))
y_noise = rng.integers(0, 2, 100)

X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_noise, cv=5).mean()

right_pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
right = cross_val_score(right_pipe, X_noise, y_noise, cv=5).mean()
print(f"selection outside CV: {wrong:.2f}   <- looks impressive, entirely fake")
print(f"selection inside CV : {right:.2f}   <- honest: no signal")

selection outside CV: 0.85   <- looks impressive, entirely fake
selection inside CV : 0.48   <- honest: no signal


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from sklearn.feature_selection import SelectKBest, f_classif</code></td><td>Import <code>SelectKBest</code> and <code>f_classif</code> from scikit-learn&#x27;s <code>feature_selection</code> module.</td></tr>
<tr><td><code>from sklearn.model_selection import cross_val_score</code></td><td>Import <code>cross_val_score</code> from scikit-learn&#x27;s <code>model_selection</code> module.</td></tr>
<tr><td><code>rng = np.random.default_rng(0)</code></td><td>Seeded generator.</td></tr>
<tr><td><code>X_noise = rng.normal(size=(100, 5000))</code></td><td>100 samples, 5000 features of pure random noise.</td></tr>
<tr><td><code>y_noise = rng.integers(0, 2, 100)</code></td><td>Random labels: no real signal exists. Honest accuracy should be ~0.5.</td></tr>
<tr><td><code>X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)</code></td><td>WRONG: picks the 20 features most correlated with y using ALL samples.</td></tr>
<tr><td><code>wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_noise, cv=5).mean()</code></td><td>CV on the pre-selected features.</td></tr>
<tr><td><code>right_pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))</code></td><td>RIGHT: selection re-done inside each training fold.</td></tr>
<tr><td><code>right = cross_val_score(right_pipe, X_noise, y_noise, cv=5).mean()</code></td><td>CV where selection happens within each training fold.</td></tr>
<tr><td><code>print(f&quot;selection outside CV: {wrong:.2f}   &lt;- looks impressive, entirely fake&quot;)</code></td><td>The leaky estimate.</td></tr>
<tr><td><code>print(f&quot;selection inside CV : {right:.2f}   &lt;- honest: no signal&quot;)</code></td><td>The honest estimate.</td></tr>
</table>
</details>

> **🐞 Bug hunt** · **My cross-validation score is great but the model fails on new data**
>
> Classic leakage. Any step that learns from data (scaling, imputing, encoding with target information, feature selection, oversampling such as SMOTE, PCA) must happen **inside** the cross-validation loop, which means inside a Pipeline. Also check for duplicate rows across splits and for features that are computed from the target.

## 20.7 Feature engineering · *Intermediate*

> **📖 Definitions**
>
> **Feature engineering**: creating new, more informative features from raw data, such as ratios, logs, interactions, date parts and counts.

**Creating features inside a pipeline**

In [9]:
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures, KBinsDiscretizer, SplineTransformer

log_tf = FunctionTransformer(np.log1p, feature_names_out="one-to-one")
print(log_tf.fit_transform(np.array([[0.0], [9.0], [99.0]])).ravel().round(3))
poly = PolynomialFeatures(degree=2, include_bias=False)
print(poly.fit_transform([[2, 3]]), poly.get_feature_names_out(["a", "b"]))
bins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile", quantile_method="averaged_inverted_cdf")
print(bins.fit_transform(np.array([[1], [2], [3], [10], [20], [30]])).ravel())
spl = SplineTransformer(n_knots=4, degree=3)
print(spl.fit_transform(np.linspace(0, 1, 5).reshape(-1, 1)).shape)

[0.    2.303 4.605]
[[2. 3. 4. 6. 9.]] ['a' 'b' 'a^2' 'a b' 'b^2']
[0. 0. 1. 1. 2. 2.]
(5, 6)


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures, KBinsDiscretizer, SplineTransformer</code></td><td>Import <code>FunctionTransformer</code>, <code>PolynomialFeatures</code>, <code>KBinsDiscretizer</code> and <code>SplineTransformer</code> from scikit-learn&#x27;s <code>preprocessing</code> module.</td></tr>
<tr><td><code>log_tf = FunctionTransformer(np.log1p, feature_names_out=&quot;one-to-one&quot;)</code></td><td>Wrap any function as a transformer (log1p = log(1 + x), safe at 0).</td></tr>
<tr><td><code>print(log_tf.fit_transform(np.array([[0.0], [9.0], [99.0]])).ravel().round(3))</code></td><td>log1p of 0, 9 and 99: 0, 2.303, 4.605.</td></tr>
<tr><td><code>poly = PolynomialFeatures(degree=2, include_bias=False)</code></td><td>Adds squares and pairwise interactions.</td></tr>
<tr><td><code>print(poly.fit_transform([[2, 3]]), poly.get_feature_names_out([&quot;a&quot;, &quot;b&quot;]))</code></td><td>[a, b, a², ab, b²] = [2, 3, 4, 6, 9].</td></tr>
<tr><td><code>bins = KBinsDiscretizer(n_bins=3, encode=&quot;ordinal&quot;, strategy=&quot;quantile&quot;, quantile_method=&quot;averaged_inverted_cdf&quot;)</code></td><td>Equal-frequency bins.</td></tr>
<tr><td><code>print(bins.fit_transform(np.array([[1], [2], [3], [10], [20], [30]])).ravel())</code></td><td>Each value replaced by its bin number.</td></tr>
<tr><td><code>spl = SplineTransformer(n_knots=4, degree=3)</code></td><td>Smooth non-linear basis for one feature.</td></tr>
<tr><td><code>print(spl.fit_transform(np.linspace(0, 1, 5).reshape(-1, 1)).shape)</code></td><td>One input column becomes several spline basis columns.</td></tr>
</table>
</details>

**A custom transformer class**

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin

class RatioFeatures(BaseEstimator, TransformerMixin):
    """Add the ratio of two columns as a new column."""
    def __init__(self, num="bmi", den="age"):
        self.num, self.den = num, den
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X[f"{self.num}_per_{self.den}"] = X[self.num] / X[self.den]
        return X

print(RatioFeatures().fit_transform(ins[["age", "bmi"]].head(3)).round(3))

   age   bmi  bmi_per_age
0   58  27.9        0.481
1   26  39.4        1.515
2   19  34.6        1.821


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>from sklearn.base import BaseEstimator, TransformerMixin</code></td><td>Import <code>BaseEstimator</code> and <code>TransformerMixin</code> from scikit-learn&#x27;s <code>base</code> module.</td></tr>
<tr><td><code>class RatioFeatures(BaseEstimator, TransformerMixin):</code></td><td>Inherit the standard API (fit_transform, get_params, ...).</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;&quot;&quot;Add the ratio of two columns as a new column.&quot;&quot;&quot;</code></td><td>Docstring.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;def __init__(self, num=&quot;bmi&quot;, den=&quot;age&quot;):</code></td><td>Store hyperparameters only; no logic in __init__.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;self.num, self.den = num, den</code></td><td>Store the column names unchanged (scikit-learn convention).</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;def fit(self, X, y=None):</code></td><td>Define the method <code>fit</code>; besides <code>self</code> (the object) it takes <code>X</code> and <code>y</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;return self</code></td><td>Nothing to learn; must still return self.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;def transform(self, X):</code></td><td>Define the method <code>transform</code>; besides <code>self</code> (the object) it takes <code>X</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;X = X.copy()</code></td><td>Never modify the caller&#x27;s DataFrame.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;X[f&quot;{self.num}_per_{self.den}&quot;] = X[self.num] / X[self.den]</code></td><td>New column, e.g. bmi_per_age.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;return X</code></td><td>Send <code>X</code> back to the caller.</td></tr>
<tr><td><code>print(RatioFeatures().fit_transform(ins[[&quot;age&quot;, &quot;bmi&quot;]].head(3)).round(3))</code></td><td>Use it like any transformer.</td></tr>
</table>
</details>

## 20.8 Inspecting and displaying pipelines · *Intermediate*

**get_params, set_params and the step__param syntax**

In [11]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
print(list(pipe.get_params().keys())[-6:])
pipe.set_params(logisticregression__C=0.1)
print(pipe.named_steps["logisticregression"].C)
print(pipe)

['logisticregression__penalty', 'logisticregression__random_state', 'logisticregression__solver', 'logisticregression__tol', 'logisticregression__verbose', 'logisticregression__warm_start']
0.1
Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(C=0.1, max_iter=1000))])


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))</code></td><td>A two-step pipeline.</td></tr>
<tr><td><code>print(list(pipe.get_params().keys())[-6:])</code></td><td>Every nested hyperparameter is addressable as step__param.</td></tr>
<tr><td><code>pipe.set_params(logisticregression__C=0.1)</code></td><td>Double underscore reaches inside a step. Grid search uses exactly this.</td></tr>
<tr><td><code>print(pipe.named_steps[&quot;logisticregression&quot;].C)</code></td><td>Display the value of <code>pipe.named_steps[&quot;logisticregression&quot;].C</code>.</td></tr>
<tr><td><code>print(pipe)</code></td><td>Text summary. In Jupyter, pipelines display as an interactive diagram.</td></tr>
</table>
</details>

> **✍️ Try it yourself**
>
> 1. Build a pipeline for the insurance data that log-transforms `charges` as the target with `TransformedTargetRegressor(regressor=..., func=np.log1p, inverse_func=np.expm1)`. Does test R² improve?
> 2. Add `PolynomialFeatures(2)` to the numeric branch. What happens to the number of features?
> 3. Repeat the leakage experiment with `k=5` and `k=200`. How does the fake score change?
>
> *Worked solutions:* [solutions-parts-5-8.ipynb](../../solutions/solutions-parts-5-8.ipynb), section *Chapter 20*.

---

⬅️ [Chapter 19: Machine-learning concepts and the scikit-learn API](19-sklearn-concepts.ipynb) · [📚 Contents](../../book/README.md) · [Chapter 21: Regression: predicting numbers](21-sklearn-regression.ipynb) ➡️